In [19]:
!pip install transformers datasets accelerate -q

In [20]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

urllib.request.urlretrieve(url,"input.txt")

with open("input.txt", "r") as f:
    text = f.read()

print(f"Total characters: {len(text)}")
print(text[:500])

Total characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [21]:
from transformers import GPT2Tokenizer,GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")
model = model.to("cuda")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [22]:
def chunk_text(text, tokenizer, block_size=128):
  tokens = tokenizer.encode(text)
  chunks = []
  for i in range(0,len(tokens)-block_size,block_size):
    chunks.append(tokens[i:i+block_size])
  return chunks

chunks = chunk_text(text, tokenizer)
print(f"Number of training chunks: {len(chunks)}")
print(f"Example chunk: {chunks[0]}")
print(f"Decoded example chunk: {tokenizer.decode(chunks[0])}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (338025 > 1024). Running this sequence through the model will result in indexing errors


Number of training chunks: 2640
Example chunk: [5962, 22307, 25, 198, 8421, 356, 5120, 597, 2252, 11, 3285, 502, 2740, 13, 198, 198, 3237, 25, 198, 5248, 461, 11, 2740, 13, 198, 198, 5962, 22307, 25, 198, 1639, 389, 477, 12939, 2138, 284, 4656, 621, 284, 1145, 680, 30, 198, 198, 3237, 25, 198, 4965, 5634, 13, 12939, 13, 198, 198, 5962, 22307, 25, 198, 5962, 11, 345, 760, 327, 1872, 385, 1526, 28599, 318, 4039, 4472, 284, 262, 661, 13, 198, 198, 3237, 25, 198, 1135, 760, 470, 11, 356, 760, 470, 13, 198, 198, 5962, 22307, 25, 198, 5756, 514, 1494, 683, 11, 290, 356, 1183, 423, 11676, 379, 674, 898, 2756, 13, 198, 3792, 470, 257, 15593, 30, 198, 198, 3237, 25, 198, 2949, 517, 3375, 319, 470, 26, 1309, 340, 307]
Decoded example chunk: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We 

In [23]:
from torch.utils.data import Dataset as TorchDataset

class ShakespeareDataset(TorchDataset):
    def __init__(self, chunks):
        self.chunks = chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        input_ids = torch.tensor(self.chunks[idx])
        return {"input_ids": input_ids, "labels": input_ids.clone()}

dataset = ShakespeareDataset(chunks)
print(f"Dataset size: {len(dataset)}")
print(dataset[0])

Dataset size: 2640
{'input_ids': tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597,  2252,    11,
         3285,   502,  2740,    13,   198,   198,  3237,    25,   198,  5248,
          461,    11,  2740,    13,   198,   198,  5962, 22307,    25,   198,
         1639,   389,   477, 12939,  2138,   284,  4656,   621,   284,  1145,
          680,    30,   198,   198,  3237,    25,   198,  4965,  5634,    13,
        12939,    13,   198,   198,  5962, 22307,    25,   198,  5962,    11,
          345,   760,   327,  1872,   385,  1526, 28599,   318,  4039,  4472,
          284,   262,   661,    13,   198,   198,  3237,    25,   198,  1135,
          760,   470,    11,   356,   760,   470,    13,   198,   198,  5962,
        22307,    25,   198,  5756,   514,  1494,   683,    11,   290,   356,
         1183,   423, 11676,   379,   674,   898,  2756,    13,   198,  3792,
          470,   257, 15593,    30,   198,   198,  3237,    25,   198,  2949,
          517,  3375,   319,   

In [24]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./gpt2-shakespeare",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    save_steps=200,
    save_total_limit=2,
    logging_steps=20,
    fp16=True,
)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

In [25]:
def generate_text(model, tokenizer, prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(
        **inputs,
        max_length=max_length,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

prompt = "To be, or not to be"
print("BEFORE FINE-TUNING:")
print(generate_text(model, tokenizer, prompt))

BEFORE FINE-TUNING:
To be, or not to be, the only way to be the only way to have the same sex, it is not necessary to have the same sex all the time, because this is the only way to have a different sex. If we have a different sex at the beginning of our lives, we will have no more sex. If we have a different sex at the end of our lives, we will have different sex in many different ways.

What's wrong with people who are obsessed


In [26]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
20,4.248265
40,3.965295
60,3.831535
80,3.752620
100,3.625492
120,3.662329
140,3.638302
160,3.657279
180,3.644273
200,3.574372


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=990, training_loss=3.4634616813274346, metrics={'train_runtime': 327.298, 'train_samples_per_second': 24.198, 'train_steps_per_second': 3.025, 'total_flos': 517358223360000.0, 'train_loss': 3.4634616813274346, 'epoch': 3.0})

In [27]:
print("\n--- AFTER FINE-TUNING ---")
print(generate_text(model, tokenizer, prompt))



--- AFTER FINE-TUNING ---
To be, or not to be:
This is the word of the king, for he was the king's son:
The king that is his son is his son's son:
And the king that was his son hath the king's son's son:
So we say, that both are sons' sons.

DUKE VINCENTIO:
I will hence unto the city: therefore, take the news.

HENRY BOLINGBRO


In [28]:
import shutil
from google.colab import files

trainer.save_model("./gpt2-shakespeare-final")
tokenizer.save_pretrained("./gpt2-shakespeare-final")
shutil.make_archive("gpt2-shakespeare-final", 'zip', "./gpt2-shakespeare-final")
files.download("gpt2-shakespeare-final.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>